# 8. Skinning

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

Linear blend skinning (LBS) decides which joints each vertex follows. Forward kinematics has already produced a new position `p_i` and a new orientation `R_i` for every joint. The rest joints are still `J[i]`.

## One 4×4 matrix per joint

The transform that takes a point in the rest pose onto the posed hand is

```text
T_i = [[ R_i ,  p_i - R_i @ J[i] ],
       [ 0   ,  1               ]]
```

Joint `i` alone sends a rest position `v` to

```text
R_i @ (v - J[i]) + p_i
```

That expression moves the point to the joint centre, rotates it, and puts it at the joint's new position.

The final position of vertex `v` is the weighted average over joints:

```text
vertices[v] = Σ_i  weight[v, i] * (T_i applied to v_posed[v])
```

`weights` has shape `(778, 16)`. Row `v` is the weight of vertex `v` on the 16 bones, and the row sums to 1. A row that is a single 1 and fifteen 0s glues that vertex to one bone. A row split into 0.25 and 0.75 lands between the two positions those bones would have assigned.

## The whole calculation on the toy

Toy weights: vertices 0 through 3 are glued to the wrist, MCP, PIP, and DIP. Vertex 4 is skin, with 0.25 on the MCP and 0.75 on the PIP.

Turn the pose blend off and rotate only the PIP by 90 degrees about X. Forward kinematics should give wrist `(0,0,0)`, MCP `(0,1,0)`, PIP `(0,2,0)`, DIP `(0,2,1)`. The skin vertex rests at `(0, 2, 0.3)` and, after the blend, should be `(0, 1.775, 0.075)`.

A point exactly on the hinge has `v - J = 0`, so rotation leaves it at the joint centre. The skin in this example has to sit off the PIP, or the blend is invisible.


In [ ]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

import toy_mano as toy

pose = np.zeros((4, 3))
pose[2, 0] = np.deg2rad(90.0)
out = toy.forward(pose, use_pose_blend=False)
print("joints")
print(np.round(out["joints"], 4))
print("vertices")
print(np.round(out["vertices"], 4))
print("sum of each weight row", toy.WEIGHTS.sum(axis=1))

assert np.allclose(out["vertices"][3], out["joints"][3])
assert np.allclose(out["vertices"][4], [0.0, 1.775, 0.075])
assert not np.allclose(out["vertices"][4], out["joints"][2])
assert np.allclose(toy.WEIGHTS.sum(axis=1), 1.0)

blended = toy.forward(pose, use_pose_blend=True)
print("skin vertex with the pose blend on", np.round(blended["vertices"][4], 4))
assert np.allclose(blended["vertices"][4], [0.0, 1.7375, 0.0875])


joints
[[0. 0. 0.]
 [0. 1. 0.]
 [0. 2. 0.]
 [0. 2. 1.]]
vertices
[[0.    0.    0.   ]
 [0.    1.    0.   ]
 [0.    2.    0.   ]
 [0.    2.    1.   ]
 [0.    1.775 0.075]]
sum of each weight row [1. 1. 1. 1. 1.]
skin vertex with the pose blend on [0.     1.7375 0.0875]


With the pose blend from chapter 7 turned on, the skin is already `(0, 2, 0.35)` before skinning, and `(0, 1.7375, 0.0875)` after it.

## The joints you should report

`joints` returned by `forward` is the 16 forward-kinematics points, plus `transl`.

Multiplying the skinned vertices by `J_regressor` again produces a different set of points. Shared weights and pose blend shapes both pull the two sets apart. When you align to a sensor or a dataset, use the forward-kinematics set.

## Check

The toy skin vertex equals neither the PIP joint `(0, 2, 0)` nor the DIP joint `(0, 2, 1)`. Its weight is split between the MCP and the PIP. Three quarters of the PIP result plus one quarter of the MCP result is `(0, 1.775, 0.075)`.

The next chapter opens the real pickle and matches the array names from these three chapters.
